# IoU e Dice 

As métricas de sobreposição próprias, sanidade completa e o primeiro confronto contra as máscaras de referência.


In [2]:
import sys
from pathlib import Path

# O kernel inicia na pasta DESTE caderno — localiza a raiz do projeto:
RAIZ = Path.cwd()
if not (RAIZ / "src" / "leitura.py").exists():
    RAIZ = RAIZ.parent          # kernel começou dentro de notebooks/
sys.path.insert(0, str(RAIZ))   # raiz no path -> 'from src...' funciona

# Imports padrão de todos os cadernos:
from src.leitura import (PASTA_IMAGENS, PASTA_RESULTADOS, carregar_imagem,
                         nome_da_mascara, criar_pastas_resultados, auditar_base)
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("raiz do projeto:", RAIZ)

raiz do projeto: /home/isadora/Documentos/ProcessIMg


In [3]:
import cv2

from src.leitura import nome_da_mascara
from src.preprocessamento import para_cinza, para_hsv
from src.segmentacao import binarizar, otsu
from src.morfologia import pos_processar
from src.avaliacao import iou, dice

In [4]:
a = np.zeros((100, 100), dtype=np.uint8); a[30:70, 30:70] = 255
b_id = a.copy()
b_inv = 255 - a
b_vazia = np.zeros_like(a)
b_desloc = np.zeros_like(a); b_desloc[50:90, 50:90] = 255   # sobrepõe 1/4 do a

print(f"idênticas:      IoU={iou(a, b_id):.3f} Dice={dice(a, b_id):.3f} (esperado 1, 1)")
print(f"invertidas:     IoU={iou(a, b_inv):.3f} Dice={dice(a, b_inv):.3f} (esperado 0, 0)")
print(f"predita vazia:  IoU={iou(b_vazia, a):.3f} Dice={dice(b_vazia, a):.3f} (esperado 0, 0)")
print(f"deslocada 25%:  IoU={iou(a, b_desloc):.3f} Dice={dice(a, b_desloc):.3f} (esperado ~0,143/~0,25)")
print(f"duas vazias:    IoU={iou(b_vazia, b_vazia):.3f} (esperado 0 — sem divisão por zero)")

# propriedade matemática: Dice >= IoU SEMPRE (mesma interseção, denominador menor)
rng = np.random.default_rng(42)
ok = all(dice((p[0].astype(np.uint8)) * 255, (p[1].astype(np.uint8)) * 255)
         >= iou((p[0].astype(np.uint8)) * 255, (p[1].astype(np.uint8)) * 255) - 1e-12
         for p in ((rng.random((60, 60)) > 0.5, rng.random((60, 60)) > 0.5) for _ in range(500)))
print(f"Dice >= IoU em 500 pares aleatórios: {ok}")

idênticas:      IoU=1.000 Dice=1.000 (esperado 1, 1)
invertidas:     IoU=0.000 Dice=0.000 (esperado 0, 0)
predita vazia:  IoU=0.000 Dice=0.000 (esperado 0, 0)
deslocada 25%:  IoU=0.143 Dice=0.250 (esperado ~0,143/~0,25)
duas vazias:    IoU=0.000 (esperado 0 — sem divisão por zero)
Dice >= IoU em 500 pares aleatórios: True


In [5]:
# o confronto: máscara do pipeline vs máscara de referência (dermatologista)
print(f"{'imagem':<16} {'canal':>6} {'IoU':>7} {'Dice':>7}")
for nome, canal, usar_s, inv in [
    ("ISIC_0010206.jpg", "cinza", False, True),
    ("ISIC_0010206.jpg", "S", True, False),
    ("ISIC_0012940.jpg", "cinza", False, True),
    ("ISIC_0012940.jpg", "S", True, False),
]:
    img = carregar_imagem(PASTA_IMAGENS / nome)
    fonte = para_hsv(img)[:, :, 1] if usar_s else para_cinza(img)
    bruta = binarizar(fonte, otsu(fonte), invertida=inv)
    predita = pos_processar(bruta)
    ref = cv2.imread(str(nome_da_mascara(PASTA_IMAGENS / nome)), cv2.IMREAD_GRAYSCALE)
    print(f"{Path(nome).stem:<16} {canal:>6} {iou(predita, ref):>7.3f} {dice(predita, ref):>7.3f}")

imagem            canal     IoU    Dice
ISIC_0010206      cinza   0.764   0.866
ISIC_0010206          S   0.904   0.950
ISIC_0012940      cinza   0.034   0.066
ISIC_0012940          S   0.034   0.066
